In [1]:
import torch
from lightning.pytorch.callbacks import Callback

class VisualLoggingCallback(Callback):
    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        # Only log for the first batch of the validation epoch
        if batch_idx == 0:
            img_input, img_target = batch
            
            # Switch to eval mode for prediction
            pl_module.eval()
            with torch.no_grad():
                img_pred = pl_module(img_input.to(pl_module.device))
            pl_module.train()

            # Create a grid: Input, Prediction, and Target side-by-side
            # (Assumes images are normalized; you might need to un-normalize)
            comparison = torch.cat([img_input[:4], img_pred[:4], img_target[:4]], dim=3)
            
            # Log to your preferred logger (e.g., TensorBoard or WandB)
            trainer.logger.experiment.add_images('Visual_Progress', comparison, trainer.global_step)

# Add it to your Trainer
# trainer = L.Trainer(callbacks=[VisualLoggingCallback()])

In [2]:
import torch
import torch.nn.functional as F
import lightning as L
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

class Image2ImageModel(L.LightningModule):
    def __init__(self, model, lr=1e-4):
        super().__init__()
        self.model = model  # e.g., a timm model or custom UNet
        self.lr = lr
        
        # Metrics for I2I are specialized
        self.psnr = PeakSignalNoiseRatio()
        self.ssim = StructuralSimilarityIndexMeasure()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        # 1. Prepare Data (Input -> Target)
        img_input, img_target = batch 
        
        # 2. Forward Pass
        img_pred = self(img_input)
        
        # 3. Calculate Loss (MSE or L1 are standard for I2I)
        loss = F.mse_loss(img_pred, img_target)
        
        # 4. Logging
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img_input, img_target = batch
        img_pred = self(img_input)
        
        val_loss = F.mse_loss(img_pred, img_target)
        
        # Update and Log Metrics
        self.psnr(img_pred, img_target)
        self.ssim(img_pred, img_target)
        
        self.log("val_loss", val_loss, prog_bar=True)
        self.log("val_psnr", self.psnr, on_epoch=True)
        self.log("val_ssim", self.ssim, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)
    


In [3]:
from tqdm.auto import tqdm
import time
import torch

def validate(model, val_loader, device, scaler):
    model.eval()
    running = 0.0
    n = 0
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
                y_pred = model(x)
                loss = torch.nn.functional.mse_loss(y_pred, y, reduction="sum")
            running += loss.item()
            n += x.size(0)
    model.train()
    return running / max(n, 1)


def one_step(batch, model, device, scaler, grad_accum_steps):
    x, y = batch
    x, y = x.to(device), y.to(device)

    with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
        y_pred = model(x)
        loss = torch.nn.functional.mse_loss(y_pred, y) / grad_accum_steps

    scaler.scale(loss).backward()
    return loss.item() * grad_accum_steps

def train(
    model,
    train_loader,
    val_loader,
    epochs: int = 20,
    lr: float = 1e-4,
    device=None,
    grad_accum_steps: int = 1,
    amp: bool = True,
    scheduler_factory=None,  # function(optimizer) -> scheduler or None
    checkpoint_path: str = "checkpoint.pt",
    validate_every: int = 1,
):
    device = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = scheduler_factory(optimizer) if scheduler_factory is not None else None
    scaler = torch.amp.GradScaler('cuda', enabled=(amp and device.type == "cuda"))

    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        model.train()
        running_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}/{epochs}")
        
        optimizer.zero_grad()

        for i, batch in pbar:
            loss = one_step(batch, model, device, scaler, grad_accum_steps)

            if (i + 1) % grad_accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            running_loss += loss.item() * grad_accum_steps
            pbar.set_postfix(train_loss=running_loss / (i + 1))

        if scheduler is not None:
            try:
                scheduler.step()
            except Exception:
                pass

        val_loss = None
        if epoch % validate_every == 0 and val_loader is not None:
            val_loss = validate(model, val_loader, device, scaler)

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        }, checkpoint_path)

        epoch_time = time.time() - epoch_start
        print(f"Epoch {epoch} done in {epoch_time:.1f}s, train_loss={running_loss/len(train_loader):.6f}"
              + (f", val_loss={val_loss:.6f}" if val_loss is not None else ""))

# Example scheduler factory (optional):
# scheduler_factory = lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
# Usage:
# train(model, train_loader, val_loader, epochs=30, lr=3e-4, grad_accum_steps=2, scheduler_factory=scheduler_factory)

In [ ]:
from  models.vit import SphereEncoderViT, SphereDecoderViT, sphereify

batch_size = 2
img_size = 256
patch_size = 16
in_channels = 3
hidden_dim = 512
latent_channels = 256
num_classes = 1000
num_heads = 8
depth = 8

print("Instantiating Sphere Encoder and Decoder...")
encoder = SphereEncoderViT(
    img_size=img_size, 
    patch_size=patch_size, 
    in_channels=in_channels, 
    hidden_dim=hidden_dim, 
    latent_channels=latent_channels,
    num_classes=num_classes,
    num_heads=num_heads,
    depth=depth,
)

decoder = SphereDecoderViT(
    img_size=img_size, 
    patch_size=patch_size, 
    out_channels=in_channels, 
    hidden_dim=hidden_dim, 
    latent_channels=latent_channels,
    num_classes=num_classes,
    num_heads=num_heads,
    depth=depth,
)


Instantiating Sphere Encoder and Decoder...


In [6]:
decoder

SphereDecoderViT(
  (ffn): Linear(in_features=256, out_features=512, bias=True)
  (norm): RMSNorm((512,), eps=1e-06, elementwise_affine=True)
  (mixer_blocks): ModuleList(
    (0-3): 4 x MLPMixerBlock(
      (norm1): RMSNorm((512,), eps=1e-06, elementwise_affine=True)
      (token_mix): Mlp(
        (fc1): Linear(in_features=256, out_features=256, bias=True)
        (act): SiLU()
        (fc2): Linear(in_features=256, out_features=256, bias=True)
      )
      (norm2): RMSNorm((512,), eps=1e-06, elementwise_affine=True)
      (channel_mix): Mlp(
        (fc1): Linear(in_features=512, out_features=1024, bias=True)
        (act): SiLU()
        (fc2): Linear(in_features=1024, out_features=512, bias=True)
      )
    )
  )
  (class_embed): Embedding(1001, 512)
  (vit_blocks): ModuleList(
    (0-7): 8 x ViTBlockAdaLNZero(
      (norm1): RMSNorm((512,), eps=1e-06, elementwise_affine=True)
      (attn): AttentionRoPE(
        (qkv): Linear(in_features=512, out_features=1536, bias=True)
     